# ED Pipeline v8 — Phase 1 Baseline + Phase 2 Guarded (Notebook Patch)

**Run order:** top → bottom. Place `ed_pipeline_v8.py` (canonical) or `ed_pipeline_v8(2).py` in the working dir. No sidecars. Uses `.loc` for IO. Phase 2 remains guarded.


In [ ]:
# === Cell 0: Canonical module bind (no side effects) ===
from pathlib import Path
import runpy

mod = None
if Path("./ed_pipeline_v8.py").exists():
    try:
        mod = runpy.run_path("./ed_pipeline_v8.py")
    except Exception as e:
        print("Note: ed_pipeline_v8.py present but could not be run via runpy:", e)
if mod is None and Path("./ed_pipeline_v8(2).py").exists():
    mod = runpy.run_path("./ed_pipeline_v8(2).py")

if mod is not None:
    WorkflowState = mod["WorkflowState"]
    TinyCritics = mod["TinyCritics"]
    CONFIG = mod["CONFIG"]

assert "WorkflowState" in globals() and "TinyCritics" in globals() and "CONFIG" in globals(), (
    "WorkflowState/TinyCritics/CONFIG must be defined by earlier cells or by this alignment."
)
print("OK: canonical symbols bound in notebook namespace.")


In [ ]:
# === Cell A: Back-compat shim for WorkflowState.update_state_from_event ===
_WS = globals().get("WorkflowState", None)
assert _WS is not None, "WorkflowState must be defined before this cell."

if not hasattr(_WS, "update_state_from_event"):
    def _update_state_from_event(self, event: dict):
        for candidate in ("apply_event", "update_from_event", "update_from_dict", "update"):
            fn = getattr(self, candidate, None)
            if callable(fn):
                return fn(event)
        backing = getattr(self, "state", None)
        if backing is None:
            backing = {}
            setattr(self, "state", backing)
        for key, val in (event or {}).items():
            parts = str(key).split(".")
            d = backing
            for p in parts[:-1]:
                if p not in d or not isinstance(d[p], dict):
                    d[p] = {}
                d = d[p]
            d[parts[-1]] = val
        return self
    setattr(_WS, "update_state_from_event", _update_state_from_event)

_s = _WS(role="nurse")
_s.update_state_from_event({"age": 60, "vitals.sbp": 95})
print("OK: WorkflowState.update_state_from_event available.")


In [ ]:
# === Cell T0: tracker_core in-notebook (first-class, .loc IO, no sidecars) ===
import sys, types, pandas as pd, numpy as np
from pathlib import Path

assert "CONFIG" in globals(), "CONFIG must be in scope."

def _ensure_parent(path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)

def _utcnow_iso():
    return pd.Timestamp.utcnow().isoformat()

class EquipmentRepository:
    def __init__(self, path: str):
        self.path = Path(path); _ensure_parent(self.path)
        if not self.path.exists():
            pd.DataFrame(columns=["id","location","last_seen"]).to_csv(self.path, index=False)
    def read(self) -> pd.DataFrame:
        df = pd.read_csv(self.path)
        for col in ["id","location","last_seen"]:
            if col not in df.columns:
                df[col] = np.nan
        return df[["id","location","last_seen"]]
    def upsert_location(self, eq_id: str, to_loc: str):
        df = self.read()
        now = _utcnow_iso()
        if (df["id"] == eq_id).any():
            idx = df.index[df["id"] == eq_id][0]
            df.loc[idx, "location"] = to_loc
            df.loc[idx, "last_seen"] = now
        else:
            df = pd.concat([df, pd.DataFrame([{ "id": eq_id, "location": to_loc, "last_seen": now }])], ignore_index=True)
        df.to_csv(self.path, index=False)
        return df

class MovesLogRepository:
    def __init__(self, path: str):
        self.path = Path(path); _ensure_parent(self.path)
        if not self.path.exists():
            pd.DataFrame(columns=["ts","id","from","to"]).to_csv(self.path, index=False)
    def append(self, eq_id: str, frm, to):
        df = pd.read_csv(self.path)
        row = {"ts": _utcnow_iso(), "id": eq_id, "from": frm, "to": to}
        df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
        df.to_csv(self.path, index=False)
        return row

class SOPRegistry:
    def __init__(self, path: str):
        self.path = Path(path); _ensure_parent(self.path)
        if not self.path.exists():
            pd.DataFrame([
                {"id":"sop-triage","title":"ED Triage","url":"about:blank"},
                {"id":"sop-ecg","title":"ECG Acquisition","url":"about:blank"},
                {"id":"sop-sepsis","title":"Sepsis Bundle","url":"about:blank"},
                {"id":"sop-stemi","title":"STEMI Activation","url":"about:blank"},
                {"id":"sop-airway","title":"Difficult Airway","url":"about:blank"},
            ]).to_csv(self.path, index=False)
    def read(self) -> pd.DataFrame | None:
        try:
            return pd.read_csv(self.path)
        except Exception:
            return None

class QRService:
    def __init__(self, out_dir: str):
        self.out_dir = Path(out_dir); self.out_dir.mkdir(parents=True, exist_ok=True)
    def make(self, payload: str) -> str:
        try:
            import qrcode
            img = qrcode.make(payload)
            p = self.out_dir / f"qr_{int(pd.Timestamp.utcnow().timestamp())}.png"
            img.save(p); return str(p)
        except Exception:
            p = self.out_dir / f"qr_{int(pd.Timestamp.utcnow().timestamp())}.txt"
            p.write_text(payload); return str(p)
    def decode(self, path: str) -> str | None:
        try:
            from PIL import Image
            from pyzbar.pyzbar import decode as _decode
            res = _decode(Image.open(path))
            if res: return res[0].data.decode("utf-8", errors="ignore")
        except Exception:
            pass
        try:
            p = Path(path)
            if p.suffix.lower() == ".txt":
                return p.read_text()
        except Exception:
            pass
        return None

class TrackerService:
    def __init__(self, equipment_repo: EquipmentRepository, moves_repo: MovesLogRepository, sop_registry: SOPRegistry):
        self.equipment_repo = equipment_repo
        self.moves_repo = moves_repo
        self.sop_registry = sop_registry
    @classmethod
    def from_config(cls, CONFIG):
        return cls(
            EquipmentRepository(CONFIG["EQUIPMENT_STATUS_PATH"]),
            MovesLogRepository(CONFIG["EQUIPMENT_MOVES_LOG_PATH"]),
            SOPRegistry(CONFIG["SOP_REGISTRY_PATH"])
        )
    def equipment_status(self) -> pd.DataFrame:
        return self.equipment_repo.read()
    def log_move(self, eq_id: str, frm, to) -> dict:
        df = self.equipment_repo.read()
        current = None
        if (df["id"] == eq_id).any():
            idx = df.index[df["id"] == eq_id][0]
            current = df.loc[idx, "location"]
        self.equipment_repo.upsert_location(eq_id, to)
        return self.moves_repo.append(eq_id, current if frm is None else frm, to)

import types as _types, sys as _sys
_tracker_mod = _types.ModuleType("tracker_core")
for _name, _obj in {
    "EquipmentRepository": EquipmentRepository,
    "MovesLogRepository": MovesLogRepository,
    "SOPRegistry": SOPRegistry,
    "QRService": QRService,
    "TrackerService": TrackerService,
}.items():
    setattr(_tracker_mod, _name, _obj)
_sys.modules["tracker_core"] = _tracker_mod
print("OK: tracker_core (in-notebook) ready with first-class services.")


In [ ]:
# === Cell B: dtype-safe assignment helper (optional) ===
import numpy as np, pandas as pd
def safe_set_missing(df: pd.DataFrame, idx, col: str):
    if col not in df.columns: raise KeyError(f"Column '{col}' not found")
    if pd.api.types.is_numeric_dtype(df[col].dtype):
        df.loc[idx, col] = np.nan
    else:
        df.loc[idx, col] = None

_tmp = pd.DataFrame({"troponin_delta": pd.Series([0.1, 0.2], dtype="float64")})
safe_set_missing(_tmp, 0, "troponin_delta")
assert pd.isna(_tmp.loc[0, "troponin_delta"]) ; print("OK: safe_set_missing works.")


In [ ]:
# === Cell C: CONFIG defaults per checklist ===
assert "CONFIG" in globals(), "CONFIG must already exist."
CONFIG["RUN_UI"] = False
CONFIG["RUN_PIPELINE"] = False
print("OK: CONFIG flags set (RUN_UI=False, RUN_PIPELINE=False)")


In [ ]:
# === Cell D: Delivery checklist (2–14) ===
import pandas as pd
from pathlib import Path
from tracker_core import TrackerService, QRService, EquipmentRepository, MovesLogRepository, SOPRegistry

# 2–6 TinyCritics cold-start
s = WorkflowState(role="nurse")
getattr(s, "touch_now", lambda *_: None)(pd.Timestamp.utcnow())
tc = TinyCritics()
p, b, u = tc.score(s, [
    {"id":"reassess_vitals","label":"Reassess vitals"},
    {"id":"order_ecg","label":"Order ECG"},
])
assert len(p) == 2 and (0<=p).all() and (p<=1).all()

# 7–14 Tracker core basic IO
t = TrackerService.from_config(CONFIG)
_ = t.equipment_status()
t.log_move("pump-001","A1","B2")
assert Path(CONFIG["EQUIPMENT_MOVES_LOG_PATH"]).exists()
q = QRService(CONFIG["QR_OUTPUT_DIR"]).make("poctest")
assert isinstance(q, str) and len(q) > 0
sop = SOPRegistry(CONFIG["SOP_REGISTRY_PATH"]).read()
assert sop is not None
print("SMOKE_OK")


In [ ]:
# === Cell E: Surfaces (present & offline-safe) ===
def refresh_sop_registry(CONFIG, base_url: str):
    summary = {"ok": False, "reason": None, "updated": 0}
    try:
        import requests  # lazy; real logic delegated to service layer
        summary["reason"] = "delegated"
    except Exception as e:
        summary["reason"] = f"unavailable: {e!r}"
    return summary

def qr_scan_fallback(payload: str, tracker: "TrackerService"):
    try:
        parts = dict(kv.split("=",1) for kv in payload.split("&") if "=" in kv)
        eq_id, to_loc = parts.get("id"), parts.get("to")
        if eq_id and to_loc:
            tracker.log_move(eq_id, None, to_loc)
            return {"ok": True, "id": eq_id, "to": to_loc}
        return {"ok": False, "reason": "missing id/to"}
    except Exception as e:
        return {"ok": False, "reason": str(e)}

ALERT_THRESHOLDS_MIN = {
    "equipment_overdue": 60,
    "lingering_patient": 120,
}
print("OK: surfaces present (offline-safe).")


In [ ]:
# === Cell P2: Phase 2 guard scaffold ===
if CONFIG.get("RUN_PIPELINE"):
    print("Phase 2 enabled — add your guarded asserts here.")
else:
    print("Phase 2 disabled (set CONFIG['RUN_PIPELINE']=True to enable).")
